# BÀI THỰC HÀNH 2: DỰ ĐOÁN DOANH SỐ SẢN PHẨM — NHÓM 3
## NOTEBOOK DEMO: TRẢI NGHIỆM SUY LUẬN & DỰ ĐOÁN DOANH SỐ THỜI GIAN THỰC

> **Mục tiêu kỹ thuật:**
> - Nạp mô hình Quán quân đã lưu (`best_model.joblib` - Random Forest Regressor) và bộ tiền xử lý `preprocessor.joblib`.
> - Tái sử dụng 100% module từ `scripts/feature_engineering.py` (`prepare_sample_from_dict`).
> - Tích hợp cơ chế kiểm soát chất lượng đầu vào (Guardrails): bắt lỗi số lượng âm/rỗng hoặc dữ liệu thiếu logic.
> - Cung cấp ô tương tác tự do (Interactive Playground) cho người dùng kiểm thử đơn hàng tùy chọn.
> - Cung cấp khu vực trải nghiệm lựa chọn mô hình tùy chỉnh (Model Selection Playground) để chạy và đối chiếu trực tiếp cả 3 thuật toán Scratch (`linear_regression`, `decision_tree`, `random_forest`).

---
### 1. Nạp Mô hình Tối ưu & Thiết lập Môi trường Suy luận
- Tự động định vị thư mục gốc dự án.
- Nạp mô hình hồi quy tối ưu và siêu dữ liệu kiểm định từ `models/saved_models/`.

In [2]:
import os
import sys
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

if sys.platform == 'win32':
    try:
        sys.stdout.reconfigure(encoding='utf-8')
        sys.stderr.reconfigure(encoding='utf-8')
    except Exception:
        pass

BASE_DIR = Path.cwd().parent if Path.cwd().name in ['scripts', 'notebooks'] else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from scripts.feature_engineering import prepare_sample_from_dict

MODEL_PATH = BASE_DIR / 'models' / 'saved_models' / 'best_model.joblib'
META_PATH = BASE_DIR / 'models' / 'saved_models' / 'metadata.json'

assert MODEL_PATH.exists(), f"Chưa tìm thấy mô hình tại {MODEL_PATH}. Hãy chạy scripts/run_pipeline.py trước!"
model = joblib.load(MODEL_PATH)

meta = {}
if META_PATH.exists():
    with open(META_PATH, 'r', encoding='utf-8') as f:
        meta = json.load(f)

print(f"✓ Đã nạp thành công mô hình: {meta.get('best_model', 'Random Forest Regressor')}")

✓ Đã nạp thành công mô hình: Random Forest Regressor (Ensemble Bagging Scratch)


---
### 2. Xây dựng Hàm Suy luận Doanh số kèm Cơ chế Guardrails
- Kiểm tra tính hợp lệ của đơn hàng: số lượng $Qty > 0$, giá trị định tính không được để trống.
- Tự động bóc tách đặc trưng miền qua hàm chuẩn hóa `prepare_sample_from_dict` từ `scripts/feature_engineering.py`.
- Áp dụng nghịch đảo `np.expm1` với chặn trần/sàn chống tràn số học.

In [3]:
def predict_product_sales(order_dict: dict, verbose: bool = True) -> float:
    """
    Dự đoán doanh số bán hàng (INR) từ thông tin đơn hàng với cơ chế bảo vệ Guardrails.
    """
    qty = order_dict.get('Qty', 1)
    if qty is None or qty <= 0:
        print("⚠️ [CẢNH BÁO GUARDRAIL]: Số lượng đặt hàng (Qty) phải là số nguyên dương (> 0).")
        return 0.0

    category = order_dict.get('Category', '')
    if not category or str(category).strip() == '':
        print("⚠️ [CẢNH BÁO GUARDRAIL]: Ngành hàng sản phẩm (Category) không được để trống.")
        return 0.0

    df_sample = prepare_sample_from_dict(order_dict)

    pred_log = model.predict(df_sample)[0]
    pred_inr = float(np.clip(np.expm1(np.clip(pred_log, -10.0, 15.0)), 0.0, None))

    if verbose:
        print("=" * 60)
        print("   📦 KẾT QUẢ DỰ BÁO DOANH THU ĐƠN HÀNG (REAL-TIME PREDICTION)")
        print("=" * 60)
        print(f"  • Ngành hàng (Category):       {order_dict.get('Category', 'N/A')}")
        print(f"  • Kích cỡ (Size):              {order_dict.get('Size', 'N/A')}")
        print(f"  • Số lượng đặt mua (Qty):      {qty} sản phẩm")
        print(f"  • Kênh giao hàng:              {order_dict.get('Fulfilment', 'Merchant')} ({order_dict.get('ship-service-level', 'Standard')})")
        print(f"  • Bang nhận hàng:              {order_dict.get('ship-state', 'Other')}")
        print("-" * 60)
        print(f"  💰 DOANH SỐ DỰ BÁO:            {pred_inr:,.2f} INR (Rupee Ấn Độ)")
        print(f"  💵 Tương đương:                ~ {pred_inr * 300:,.0f} VNĐ")
        print("=" * 60)

    return pred_inr

---
### 3. Thực nghiệm Các Tình huống Thực tế
- **Tình huống 1:** Khách lẻ mua 1 áo thời trang nữ Kurta bình dân (phân khúc 300 - 500 INR).
- **Tình huống 2:** Khách mua 1 bộ trang phục cao cấp Set áo váy (phân khúc 700 - 1.200 INR).
- **Tình huống 3:** Đơn đặt hàng sỉ số lượng lớn B2B (Qty = 5).

In [4]:
print("--- THỰC NGHIỆM TÌNH HUỐNG 1: KHÁCH LẺ MUA ÁO THỜI TRANG KURTA ---")
order_1 = {
    'Category': 'Kurta',
    'Size': 'M',
    'Qty': 1,
    'Fulfilment': 'Merchant',
    'ship-service-level': 'Standard',
    'Courier Status': 'Shipped',
    'ship-state': 'MAHARASHTRA',
    'month': 4,
    'day_of_week': 5,
    'is_promoted': 0,
    'is_b2b': 0
}
pred_1 = predict_product_sales(order_1)

print()
print("--- THỰC NGHIỆM TÌNH HUỐNG 2: BỘ TRANG PHỤC CAO CẤP (SET) ---")
order_2 = {
    'Category': 'Set',
    'Size': 'L',
    'Qty': 1,
    'Fulfilment': 'Amazon',
    'ship-service-level': 'Expedited',
    'Courier Status': 'Shipped',
    'ship-state': 'KARNATAKA',
    'month': 5,
    'day_of_week': 6,
    'is_promoted': 1,
    'is_b2b': 0
}
pred_2 = predict_product_sales(order_2)

print()
print("--- THỰC NGHIỆM TÌNH HUỐNG 3: ĐƠN HÀNG SỈ KHÁCH DOANH NGHIỆP (B2B, QTY = 5) ---")
order_3 = {
    'Category': 'Set',
    'Size': 'XL',
    'Qty': 5,
    'Fulfilment': 'Merchant',
    'ship-service-level': 'Standard',
    'Courier Status': 'Shipped',
    'ship-state': 'MAHARASHTRA',
    'month': 6,
    'day_of_week': 1,
    'is_promoted': 1,
    'is_b2b': 1
}
pred_3 = predict_product_sales(order_3)

--- THỰC NGHIỆM TÌNH HUỐNG 1: KHÁCH LẺ MUA ÁO THỜI TRANG KURTA ---
   📦 KẾT QUẢ DỰ BÁO DOANH THU ĐƠN HÀNG (REAL-TIME PREDICTION)
  • Ngành hàng (Category):       Kurta
  • Kích cỡ (Size):              M
  • Số lượng đặt mua (Qty):      1 sản phẩm
  • Kênh giao hàng:              Merchant (Standard)
  • Bang nhận hàng:              MAHARASHTRA
------------------------------------------------------------
  💰 DOANH SỐ DỰ BÁO:            421.78 INR (Rupee Ấn Độ)
  💵 Tương đương:                ~ 126,533 VNĐ

--- THỰC NGHIỆM TÌNH HUỐNG 2: BỘ TRANG PHỤC CAO CẤP (SET) ---
   📦 KẾT QUẢ DỰ BÁO DOANH THU ĐƠN HÀNG (REAL-TIME PREDICTION)
  • Ngành hàng (Category):       Set
  • Kích cỡ (Size):              L
  • Số lượng đặt mua (Qty):      1 sản phẩm
  • Kênh giao hàng:              Amazon (Expedited)
  • Bang nhận hàng:              KARNATAKA
------------------------------------------------------------
  💰 DOANH SỐ DỰ BÁO:            812.07 INR (Rupee Ấn Độ)
  💵 Tương đương:                ~ 243

---
### 4. Ô tương tác tự do (Interactive Playground)
- Người dùng có thể tùy chỉnh các thuộc tính đơn hàng bên dưới để quan sát phản hồi dự đoán tức thì từ mô hình.

> 💡 **Bảng tra cứu giá trị hợp lệ để nhập thử:**
> - **Category (Ngành hàng):** `'Western Dress'` (Đầm nữ), `'Set'` (Bộ váy áo cao cấp), `'Kurta'` (Áo thời trang), `'Top'` (Áo kiểu), `'Ethnic Dress'`, `'Blouse'`, `'Bottom'`, `'Saree'`, `'Dupatta'`.
> - **Size (Kích cỡ):** `'XS'`, `'S'`, `'M'`, `'L'`, `'XL'`, `'XXL'`, `'3XL'`, `'Free'`.
> - **Qty (Số lượng):** Số nguyên dương $\ge 1$ (Ví dụ: `1`, `2`, `5`, `10`...).
> - **Fulfilment (Kênh xử lý):** `'Amazon'` (Kho Amazon FBA đóng gói), `'Merchant'` (Shop tự gửi hàng).
> - **ship-service-level (Vận chuyển):** `'Expedited'` (Giao hỏa tốc), `'Standard'` (Giao tiết kiệm tiêu chuẩn).
> - **is_promoted (Khuyến mãi):** `1` (Có áp mã giảm giá), `0` (Giá gốc không khuyến mãi).
> - **is_b2b (Khách sỉ):** `1` (Doanh nghiệp mua sỉ B2B), `0` (Khách mua lẻ cá nhân B2C).
>
> *(Lưu ý: Bạn có thể chỉ cần nhập `Category` và `Qty`, các trường khác nếu để trống hệ thống sẽ tự động bù đắp giá trị phổ thông mặc định!)*

In [ ]:
# Tùy chỉnh đơn hàng kiểm thử (bạn có thể thay đổi các giá trị bên dưới):
my_custom_order = {
    'Category': 'Western Dress',       # Ngành hàng: Western Dress (Đầm), Set (Bộ váy áo), Kurta (Áo lẻ)...
    'Size': 'S',                       # Kích cỡ: XS, S, M, L, XL, XXL, 3XL, Free
    'Qty': 1,                          # Số lượng sản phẩm đặt mua (phải >= 1)
    'Fulfilment': 'Amazon',            # Kênh kho đóng gói: Amazon (Kho FBA) hoặc Merchant (Người bán tự gửi)
    'ship-service-level': 'Expedited', # Hình thức vận chuyển: Expedited (Hỏa tốc) hoặc Standard (Tiêu chuẩn)
    'Courier Status': 'Shipped',       # Trạng thái giao vận: Shipped (Đang giao), Unshipped (Chưa giao)
    'ship-state': 'WEST BENGAL',       # Bang nhận hàng tại Ấn Độ: MAHARASHTRA, KARNATAKA, DELHI, WEST BENGAL...
    'month': 4,                        # Tháng đặt hàng (1 đến 12)
    'day_of_week': 5,                  # Ngày trong tuần (0: Thứ 2, 1: Thứ 3, 2: Thứ 4, 3: Thứ 5, 4: Thứ 6, 5: Thứ 7, 6: Chủ nhật)
    'is_promoted': 1,                  # Áp dụng khuyến mãi: 1 (Có mã giảm giá), 0 (Không có)
    'is_b2b': 0                        # Loại khách hàng: 0 (Khách mua lẻ cá nhân), 1 (Doanh nghiệp mua sỉ B2B)
}

# Dự đoán doanh thu thời gian thực
predicted_val = predict_product_sales(my_custom_order)

   📦 KẾT QUẢ DỰ BÁO DOANH THU ĐƠN HÀNG (REAL-TIME PREDICTION)
  • Ngành hàng (Category):       Western Dress
  • Kích cỡ (Size):              S
  • Số lượng đặt mua (Qty):      1 sản phẩm
  • Kênh giao hàng:              Amazon (Expedited)
  • Bang nhận hàng:              WEST BENGAL
------------------------------------------------------------
  💰 DOANH SỐ DỰ BÁO:            739.12 INR (Rupee Ấn Độ)
  💵 Tương đương:                ~ 221,737 VNĐ


---
### 5. Thử nghiệm Lựa chọn Mô hình Tùy chỉnh (Model Selection Playground)
- Cho phép người dùng chuyển đổi linh hoạt giữa cả 3 thuật toán Scratch (`Linear Regression`, `Decision Tree Regressor`, `Random Forest Regressor`) để so sánh sự khác biệt trong dự đoán.

In [6]:
available_models = {
    "1. Random Forest Regressor (Quán quân 🏆)": BASE_DIR / 'models' / 'saved_models' / 'random_forest.joblib',
    "2. Decision Tree Regressor (Á quân)":      BASE_DIR / 'models' / 'saved_models' / 'decision_tree.joblib',
    "3. Linear Regression (Đối chứng)":         BASE_DIR / 'models' / 'saved_models' / 'linear_regression.joblib'
}

test_order = {
    'Category': 'Set',
    'Size': 'M',
    'Qty': 1,
    'Fulfilment': 'Amazon',
    'ship-service-level': 'Expedited',
    'Courier Status': 'Shipped',
    'ship-state': 'MAHARASHTRA',
    'Date': '2022-05-15',
    'month': 5,
    'day_of_week': 5,
    'is_promoted': 1,
    'is_b2b': 0
}

sample_df = prepare_sample_from_dict(test_order)

print("=" * 70)
print(f"   📊 SO SÁNH DỰ BÁO GIỮA 3 THUẬT TOÁN SCRATCH TRÊN CÙNG ĐƠN HÀNG (SET, SIZE M, QTY=1)")
print("=" * 70)

for display_name, m_path in available_models.items():
    if m_path.exists():
        pipe_m = joblib.load(m_path)
        pred_log_m = pipe_m.predict(sample_df)[0]
        pred_inr_m = float(np.clip(np.expm1(np.clip(pred_log_m, -10.0, 15.0)), 0.0, None))
        print(f"  • {display_name:<45}: {pred_inr_m:,.2f} INR (~{pred_inr_m*300:,.0f} VNĐ)")
    else:
        print(f"  • {display_name:<45}: Chưa tìm thấy tệp {m_path.name}")
print("=" * 70)

   📊 SO SÁNH DỰ BÁO GIỮA 3 THUẬT TOÁN SCRATCH TRÊN CÙNG ĐƠN HÀNG (SET, SIZE M, QTY=1)
  • 1. Random Forest Regressor (Quán quân 🏆)     : 812.07 INR (~243,622 VNĐ)
  • 2. Decision Tree Regressor (Á quân)          : 813.31 INR (~243,993 VNĐ)
  • 3. Linear Regression (Đối chứng)             : 821.14 INR (~246,343 VNĐ)
